In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.formula.api import ols

In [3]:
# Pilot study data

data = {
    "name": [],
    "gender": [],
    "ethnicity": [],
    "score": []
}

# Vi bruger gennemsnittet af lav, mellem og høj
# fordi CV-niveau ikke er det, vi undersøger

pilot_data = {
    "Aisha": ["Female", "Arabic", [62, 62, 58], [88, 82, 82], [88, 88, 88]],
    "Maria": ["Female", "Danish", [58, 58, 62], [78, 78, 78], [88, 87, 82]], #korrekt
    "Peter": ["Male", "Danish", [58, 62, 58], [78, 82, 72], [87, 82, 87]], #korrekt
    "Mohammed": ["Male", "Arabic", [68, 58, 58], [78, 78, 78], [87, 88, 88]] #korrekt
}

for name in pilot_data:
    gender = pilot_data[name][0]
    ethnicity = pilot_data[name][1]
    low = pilot_data[name][2]
    medium = pilot_data[name][3]
    high = pilot_data[name][4]
    
    for i in range(3):
        average_score = (low[i] + medium[i] + high[i]) / 3
        
        data["name"].append(name)
        data["gender"].append(gender)
        data["ethnicity"].append(ethnicity)
        data["score"].append(average_score)

df = pd.DataFrame(data)

df

,name,gender,ethnicity,score
0,Aisha,Female,Arabic,79.333333
1,Aisha,Female,Arabic,77.333333
2,Aisha,Female,Arabic,76.000000
3,Maria,Female,Danish,74.666667
4,Maria,Female,Danish,74.333333
5,Maria,Female,Danish,74.000000
6,Peter,Male,Danish,74.333333
7,Peter,Male,Danish,75.333333
8,Peter,Male,Danish,72.333333
9,Mohammed,Male,Arabic,77.666667


In [4]:
model = ols("score ~ C(gender) * C(ethnicity)", data=df).fit()

anova_table = sm.stats.anova_lm(model, typ=2)

anova_table

,sum_sq,df,F,PR(>F)
C(gender),3.703704,1.0,1.793722,0.217275
C(ethnicity),17.925926,1.0,8.681614,0.018526
C(gender):C(ethnicity),1.814815,1.0,0.878924,0.375933
Residual,16.518519,8.0,NaN,NaN


In [5]:
# Residual betyder "fejlvariationen"
# Det er variationen, som modellen ikke kan forklare

ss_error = anova_table.loc["Residual", "sum_sq"]

effect_sizes = {}

for effect in ["C(gender)", "C(ethnicity)", "C(gender):C(ethnicity)"]:
    
    ss_effect = anova_table.loc[effect, "sum_sq"]
    
    partial_eta_squared = ss_effect / (ss_effect + ss_error)
    
    cohens_f = np.sqrt(partial_eta_squared / (1 - partial_eta_squared))
    
    effect_sizes[effect] = cohens_f
    
    print(effect)
    print("Partial eta squared:", round(partial_eta_squared, 3))
    print("Cohen's f:", round(cohens_f, 3))
    print()

C(gender)
Partial eta squared: 0.183
Cohen's f: 0.474

C(ethnicity)
Partial eta squared: 0.52
Cohen's f: 1.042

C(gender):C(ethnicity)
Partial eta squared: 0.099
Cohen's f: 0.331



In [6]:
effect_size = min(effect_sizes.values())

print("Smallest effect size:", round(effect_size, 3))

Smallest effect size: 0.331
